# RoadVanta AI — RDD2022 YOLOv8 Automated Cloud Training Pipeline

This Google Colab notebook trains a custom **YOLOv8** model on your road damage dataset (`rdd_india_subset.zip` or `archive (1).zip`) using **GPU Acceleration (Tesla T4 / A100)**.

### Supported Defect Classes in Indian Roads:
- `0`: **Longitudinal Crack**
- `1`: **Transverse Crack**
- `2`: **Alligator Crack**
- `3`: **Pothole** (Primary Road Hazard)
- `4`: **Road Defect Cluster / Patch**

---

### Step 1: Verify Free GPU Acceleration
Ensure you are connected to **T4 GPU** on Colab (Top right shows green RAM/Disk).

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Training on GPU: {torch.cuda.get_device_name(0)}")

### Step 2: Install Ultralytics YOLOv8

In [ ]:
!pip install -q ultralytics pyyaml

### Step 3: Locate Uploaded ZIP File (Direct Colab Upload — No Drive Needed)

In [ ]:
import os

# Check for uploaded files directly in Colab
candidates = [
    '/content/rdd_india_subset.zip',
    '/content/archive (1).zip',
    '/content/archive.zip',
    '/content/drive/MyDrive/archive (1).zip'
]

ZIP_PATH = None
for p in candidates:
    if os.path.exists(p):
        ZIP_PATH = p
        break

if ZIP_PATH:
    print(f"[SUCCESS] Found dataset archive: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/(1024*1024):.1f} MB)")
else:
    print("[!] Please drag and drop 'rdd_india_subset.zip' into Colab's left Files panel.")

### Step 4: Extract Dataset & Prepare Configuration

In [ ]:
import os, zipfile, shutil

!mkdir -p /content/dataset
print(f"Extracting {ZIP_PATH} into /content/dataset...")
!unzip -q -o "$ZIP_PATH" -d /content/dataset

# Detect configuration or create data.yaml
yaml_path = '/content/dataset/data.yaml'
if not os.path.exists(yaml_path):
    # Check if RDD_SPLIT folder exists
    if os.path.exists('/content/dataset/RDD_SPLIT'):
        base_dir = '/content/dataset/RDD_SPLIT'
    else:
        base_dir = '/content/dataset'
        
    yaml_content = f'''
path: {base_dir}
train: train/images
val: val/images

names:
  0: longitudinal_crack
  1: transverse_crack
  2: alligator_crack
  3: pothole
  4: road_defect_cluster
'''
    with open('/content/dataset/data.yaml', 'w') as f:
        f.write(yaml_content.strip())

print("[SUCCESS] Dataset ready with YAML config at /content/dataset/data.yaml")

### Step 5: Start High-Speed GPU Training (50 Epochs)

In [ ]:
from ultralytics import YOLO

# Load baseline YOLOv8 small or nano for transfer learning
model = YOLO('yolov8s.pt')

# Train with GPU acceleration
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='roadvanta_yolov8_trained',
    patience=10,
    save=True,
    device=0
)

### Step 6: Validate Detection Accuracy (mAP50 & Precision)

In [ ]:
metrics = model.val()
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

### Step 7: Download `best.pt` to Put into RoadVanta
Once downloaded, simply place `best.pt` into `prism_backend/models/best.pt` in your project folder!

In [ ]:
from google.colab import files

weights_path = 'runs/detect/roadvanta_yolov8_trained/weights/best.pt'
print(f"Downloading {weights_path} to your computer...")
files.download(weights_path)